In [6]:
import pandas as pd
import re

# ==================== 配置 ====================
input_files = {
    '佳能': 'CanonR50.xlsx',
    '富士': 'FujiXM5.xlsx',
    '尼康': 'NikonZ30.xlsx',
    '索尼': 'SonyZVE10.xlsx'
}
output_file = 'AllCameras_Combined.xlsx'
# =============================================

# 最简单的去emoji函数
def remove_emoji_simple(text):
    if pd.isna(text):
        return text
    # 用正则去掉所有4字节以上的Unicode字符（基本就是emoji）
    return re.sub(r'[^\u0000-\uFFFF]', '', str(text))

# 读取合并
print("正在读取文件...")
dfs = []
for brand, f in input_files.items():
    df = pd.read_excel(f)
    df['品牌'] = brand
    print(f"{brand}: {len(df)}条")
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print(f"\n合并后总条数：{len(df)}")

# 重命名列（根据你的数据）
df.columns = ['用户ID', '时间', '商品规格', '评论内容', '页面标题', '品牌']

# 1. 删掉未填写评价内容
before = len(df)
df = df[~df['评论内容'].str.contains('此用户未填写评价内容', na=False)]
print(f"删除未填写评价内容：{before - len(df)}条")

# 2. 去emoji
import emoji

def remove_emoji_advanced(text):
    if pd.isna(text):
        return text
    # 这个方法能识别几乎所有emoji
    return emoji.replace_emoji(str(text), replace='')

# 替换原来的remove_emoji_simple
df['评论内容'] = df['评论内容'].apply(remove_emoji_advanced)

# 3. 去重（只去掉完全一样的）
before = len(df)
df = df.drop_duplicates(subset=['评论内容'], keep='first')
print(f"去掉完全重复的评论：{before - len(df)}条")

# 4. 按品牌排序
df = df.sort_values(['品牌', '时间'], ascending=[True, False])

# 保存
df.to_excel(output_file, index=False)
print(f"\n✅ 完成！共 {len(df)} 条")
print("\n各品牌数量：")
print(df['品牌'].value_counts())

正在读取文件...
佳能: 1288条
富士: 889条
尼康: 1062条
索尼: 941条

合并后总条数：4180
删除未填写评价内容：4条
去掉完全重复的评论：6条

✅ 完成！共 4170 条

各品牌数量：
品牌
佳能    1286
尼康    1062
索尼     940
富士     882
Name: count, dtype: int64
